In [68]:
import pandas as pd
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
import string

In [69]:
df_details = pd.read_csv("dw_data/all-detailsepisodes.csv") # columns: episodeid, title, first_diffusion, doctorid
df_imdb = pd.read_csv("dw_data/imdb_details.csv") # columns: number, title, rating, nbr_votes, description, season
# Merge datasets on title
df = pd.merge(df_details, df_imdb, on="title", how="inner")

In [70]:
def preprocess(text):
    if pd.isna(text):
        return []
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in string.punctuation]
    return tokens

In [71]:
from collections import defaultdict

inverted_index = defaultdict(list)
document_corpus = {}

# Preparing the corpus for building the inverted index
descriptions = df['description'].fillna("").tolist()
episode_keys = (df['season'].astype(str) + "x" + df['number'].astype(str)).tolist()
titles = df['title'].tolist()

# Filling the index
for i, doc in enumerate(descriptions):
    doc_id = episode_keys[i] # unique identifier for each episode, e.g., "1x01"
    
    document_corpus[doc_id] = {
        "title": titles[i],
        "description": doc,
        "id": doc_id
    }
    tokens = preprocess(doc) 
    
    for token in tokens:
        if doc_id not in inverted_index[token]:
            inverted_index[token].append(doc_id)

print(f"Index built with {len(inverted_index)} unique terms for {len(document_corpus)} episodes.")

Index built with 1512 unique terms for 135 episodes.


What data is relevant for the task of finding similar episodes based on descriptions?
What data is relevant for finding an episode suitable for user query?

The episode descriptions from the dw_imdb dataset are relevant for both tasks, as they provide textual information about each episode that can be used to compute similarity. They are also more useful since we'll be only working with the series reboot starting 2005 and not the "old" Doctor Who.

The scripts from the dw_scripts dataset may also be relevant if we want to analyze the content of the episodes in more detail, but for a basic similarity comparison, the descriptions should suffice.

In [115]:
def boolean_search(query: str, index: dict):
    query_tokens = preprocess(query)
    if not query_tokens:
      return []
    results = set(index.get(query_tokens[0], []))

    for token in query_tokens[1:]:
      results = results.union(index.get(token, []))

    return list(results)[:5]

In [109]:
from rank_bm25 import BM25Okapi

# Prepare corpus for BM25
tokenized_corpus = [preprocess(doc['description']) for doc in document_corpus.values()]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, n=5):
    query_tokens = preprocess(query)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    results = []
    for i in ranked_indices[:n]:
        doc_id = list(document_corpus.keys())[i]
        results.append({
            'episodeid': doc_id,
            'title': document_corpus[doc_id]['title'],
            'score': scores[i],
            'snippet': document_corpus[doc_id]['description'][:300]  # first 300 chars of episode
        })
    return results

In [144]:
my_queries = ["Doctor fights with Weeping Angels and wants to save Amy",
              "Doctor and Clara are in 19th century",
              "Doctor meet River Song for the first time",
              "Doctor and Donna encounter Daleks and Davros",
              "Doctor and Rose end up in a parallel universe",
              "Doctor meets van Gogh",
              "Doctor and Martha face time paradox creatures or similar threats",
              "Doctor and Bill encounter Cybermen",
              "Doctor meets Paternoster Gang, Jenny and Vastra and Strax",
              "Doctor and Rose meet her father Pete Tyler"
              ]

In [145]:
BOLD = "\033[1m"
RESET = "\033[0m"

for query in my_queries:
    print(f"\n{BOLD}Query '{query}'{RESET}")
    hits = boolean_search(query, inverted_index)
    print(f"\nBoolean hits:")
    for i, id in enumerate(hits):
        print(f"{i+1}: {id} - {document_corpus[id]['title']}\n{document_corpus[id]['description']}")
    print("\nBM25 Results:")
    results = bm25_search(query, n=5)
    for i, res in enumerate(results):
        print(f"Rank {i+1}: {res['episodeid']} - {res['title']} (score={res['score']:.2f})")
        print(res['snippet'])
        print("*-----" * 50)


Query 'Doctor fights with Weeping Angels and wants to save Amy'

Boolean hits:
1: 6x11 - The God Complex
The Doctor, Amy and Rory become trapped in a hotel of horrors unable to escape and unable to find the TARDIS. The Doctor must save as many people as he can taking many twists and seeing his own worst fear.
2: 9x7 - The Zygon Invasion
The fragile peace between the Humans and the Zygons is on a knife edge. Tensions run high as factions within the Zygon community seek to incite violent action against Humans. Called in by UNIT, the Doctor and Clara fight to bring the situation under control. But one question remains: Where on Earth is Osgood?
3: 8x10 - In the Forest of the Night
One morning in London, and every city and town in the world, the human race wakes up to the most surprising invasion yet: the trees have moved back in. Everywhere, in every land, a forest has grown and taken back the Earth.
4: 5x10 - Vincent and the Doctor
The Doctor and Amy travel back in time to meet Vincent 